### Tools

Models can request to call tools that perrform tasks such as fetching data from database, searching the web, or running code. Tools are pairings of:

    1. A Schema, including the name of the tool, a description, and/or argument definations (afte a JSON schema)
    2. A function or corountine to execute

In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"]= os.getenv("GOOGLE_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("Who is Harsh Jyoriya")
response

AIMessage(content="<think>\nOkay, the user is asking about Harsh Jyoriya. Let me start by recalling any information I have about this person. I know that sometimes people might ask about individuals who aren't widely known, so I need to check if there's public information available.\n\nFirst, I'll consider possible fields where someone might be notable. Maybe technology, entertainment, business, or academia. The name doesn't ring a bell immediately. Let me think if there's any recent news or social media presence associated with this name.\n\nI should check if there are any public profiles on LinkedIn, Twitter, or other platforms. Sometimes people gain recognition through online content, so maybe a YouTuber or blogger? Alternatively, could it be a typo? Maybe the user meant a similar name, like Harsh Jyoti or another variation.\n\nI can also think about Indian names, as the surname Jyoriya might be of Indian origin. Are there any prominent Indian figures with that name? I'm not recalli

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the wheather at a location"""
    return f"It's sunny in {location}"

models_with_tool=model.bind_tools([get_weather])

In [6]:
response=models_with_tool.invoke("What is the wheather in Jhansi")
print(response)

for tool_call in response.tool_calls:
    # View tool call made by model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Jhansi. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Jhansi is a city in India, so I should use that as the location. I need to make sure the parameters are correctly formatted. The required field is location, and the type is string. So the arguments should be {"location": "Jhansi"}. Just need to call the get_weather function with that argument.\n', 'tool_calls': [{'id': 'aa6tj4vt4', 'function': {'arguments': '{"location":"Jhansi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 156, 'total_tokens': 283, 'completion_time': 0.198055235, 'completion_tokens_details': {'reasoning_tokens': 101}, 'prompt_time': 0.007967897, 'prompt_tokens_details': None, 'queue_time': 0.445997389, 'total_time': 0.206023132}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint':

## Tool Execution Loop

In [9]:
# Step1: Model generate tool call
message = [{"role":"user", "content":"What is the wheather in Jhansi"}]
ai_msg = models_with_tool.invoke(message)
message.append(ai_msg)

# Step2: Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

# Step3: Pass result back to model for final response
final_response=models_with_tool.invoke(message)
print(final_response.text) 

The weather in Jhansi is currently sunny. 😊


In [10]:
message

[{'role': 'user', 'content': 'What is the wheather in Jhansi'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Jhansi. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Jhansi is a city in India, so I should use that function. I need to make sure the location is specified correctly. The parameters require a string, so I\'ll input "Jhansi" as the location. I\'ll call the get_weather function with that argument.\n', 'tool_calls': [{'id': 'r9x66dfty', 'function': {'arguments': '{"location":"Jhansi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 156, 'total_tokens': 273, 'completion_time': 0.191633727, 'completion_tokens_details': {'reasoning_tokens': 91}, 'prompt_time': 0.007663232, 'prompt_tokens_details': None, 'queue_time': 0.051055768, 'total_time': 0.199296959}, 'model_name': 'qwen/qwen3